In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle

In [2]:
def check_for_nonnumeric(pd_series=None):
    if pd.to_numeric(pd_series, errors='coerce').isna().sum() == 0:
        return 0
    else:
        return 1


def gene_plot(d, geneid, lfc, lfc_thr, pv_thr, genenames, gfont, pv, gstyle):
    if genenames is not None and genenames == "deg":
        for i in d[geneid].unique():
            if (d.loc[d[geneid] == i, lfc].iloc[0] >= lfc_thr[0] and d.loc[d[geneid] == i, pv].iloc[0] < pv_thr[0]) or \
                    (d.loc[d[geneid] == i, lfc].iloc[0] <= -lfc_thr[1] and d.loc[d[geneid] == i, pv].iloc[0] < pv_thr[1]):
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0], i,
                                  fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(i, xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)
    elif genenames is not None and type(genenames) is tuple:
        for i in d[geneid].unique():
            if i in genenames:
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0], i,
                                  fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(i, xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)
    elif genenames is not None and type(genenames) is dict:
        for i in d[geneid].unique():
            if i in genenames:
                if gstyle == 1:
                    plt.text(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0],
                                  genenames[i], fontsize=gfont)
                elif gstyle == 2:
                    plt.annotate(genenames[i], xy=(d.loc[d[geneid] == i, lfc].iloc[0], d.loc[d[geneid] == i, 'logpv_add_axy'].iloc[0]),
                                 xycoords='data', xytext=(5, -15), textcoords='offset points', size=6,
                                 bbox=dict(boxstyle="round", alpha=0.1),
                                 arrowprops=dict(arrowstyle="wedge,tail_width=0.5", alpha=0.1, relpos=(0, 0)))
                else:
                    print("Error: invalid gstyle choice")
                    sys.exit(1)


def volcano(df="dataframe", lfc=None, pv=None, lfc_thr=(1, 1), pv_thr=(0.05, 0.05), color=("green", "grey", "red"),
            valpha=1, geneid=None, genenames=None, gfont=8, dim=(5, 5), ar=90, dotsize=1, markerdot="o",
            sign_line_v=False, sign_line_h=False, gstyle=1, axtickfontsize=9,
            axtickfontname="Arial", axlabelfontsize=9, axlabelfontname="Arial", axxlabel=None,
            axylabel=None, xlm=None, ylm=None, plotlegend=False, legendpos='best',
            figname='volcano', legendanchor=None,
            legendlabels=['Significant up', 'Not significant', 'Significant down'], theme=None, path='', ret=False, **kwargs):
    _x = r'$ \log_{2}(\mathrm{Fold Change})$'
    _y = r'$ -\log_{10}(\mathrm{p-value})$'
    color = color
    ax = kwargs.get('ax')
    if ax:
        plt.sca(ax)
    # check if dataframe contains any non-numeric character
    assert check_for_nonnumeric(df[lfc]) == 0, 'dataframe contains non-numeric values in lfc column'
    assert check_for_nonnumeric(df[pv]) == 0, 'dataframe contains non-numeric values in pv column'
    # this is important to check if color or logpv exists and drop them as if you run multiple times same command
    # it may update old instance of df
    df = df.drop(['color_add_axy', 'logpv_add_axy'], axis=1, errors='ignore')
    assert len(set(color)) == 3, 'unique color must be size of 3'
    df.loc[(df[lfc] >= lfc_thr[0]) & (df[pv] < pv_thr[0]), 'color_add_axy'] = color[0]  # upregulated
    #df.loc[(df[lfc] >= lfc_thr[0]) & (df[pv] < pv_thr[0]), 'size_add_axy'] = dotsize[0]
    df.loc[(df[lfc] <= -lfc_thr[1]) & (df[pv] < pv_thr[1]), 'color_add_axy'] = color[2]  # downregulated
    #df.loc[(df[lfc] <= -lfc_thr[1]) & (df[pv] < pv_thr[1]), 'size_add_axy'] = dotsize[2]
    df['color_add_axy'].fillna(color[1], inplace=True)  # intermediate
    #df['size_add_axy'].fillna(dotsize[1], inplace=True)  # intermediate
    df['logpv_add_axy'] = -(np.log10(np.array(df[pv].values.astype(float))))
    # plot
    assign_values = {col: i for i, col in enumerate(color)}
    color_result_num = [assign_values[i] for i in df['color_add_axy']]

    #assert len(set(color_result_num)) == 3, \
    #    'either significant or non-significant genes are missing; try to change lfc_thr or pv_thr to include ' \
    #    'both significant and non-significant genes'
    if theme == 'dark':
        plt.style.use('dark_background')
    #plt.subplots(figsize=dim)
    if plotlegend:
        s = plt.scatter(df[lfc], df['logpv_add_axy'], c=color_result_num, cmap=ListedColormap(color), alpha=valpha,
                        s=dotsize, marker=markerdot)
        assert len(legendlabels) == 3, 'legendlabels must be size of 3'
        plt.legend(handles=s.legend_elements()[0], labels=legendlabels, loc=legendpos, bbox_to_anchor=legendanchor)
    else:
        plt.scatter(df[lfc], df['logpv_add_axy'], c=color_result_num, cmap=ListedColormap(color), alpha=valpha,
                    s=dotsize, marker=markerdot)
    if sign_line_h:
        plt.axhline(y=-np.log10(pv_thr[0]), linestyle='--', color='black', linewidth=1)
    if sign_line_v:
        plt.axvline(x=lfc_thr[0], linestyle='--', color='black', linewidth=1)
        plt.axvline(x=-lfc_thr[1], linestyle='--', color='black', linewidth=1)
    gene_plot(df, geneid, lfc, lfc_thr, pv_thr, genenames, gfont, pv, gstyle)

    if axxlabel:
        _x = axxlabel
    if axylabel:
        _y = axylabel

    plt.xlabel(_x, fontsize=axlabelfontsize, fontname=axlabelfontname)
    plt.ylabel(_y, fontsize=axlabelfontsize, fontname=axlabelfontname)
    if xlm:
        plt.xlim(left=xlm[0], right=xlm[1])
        plt.xticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)

    else:
        plt.xticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    if ylm:
        plt.ylim(bottom=ylm[0], top=ylm[1])
        plt.yticks(np.arange(ylm[0], ylm[1], ylm[2]),  fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    else:
        plt.yticks(fontsize=axtickfontsize, rotation=ar, fontname=axtickfontname)
    
    if ret:
        return plt.gca()
    else:
        plt.savefig(f"{path}/{figname}.png", bbox_inches='tight', dpi=400)
        plt.savefig(f"{path}/{figname}.pdf", bbox_inches='tight', dpi=400)
        plt.clf()
        plt.close()

In [ ]:
path_curr = "D:/Yandex.Disk/pydnameth/draft/13_fmba_cvd_dnam/data/56_vs_56"
   
df_dmps = pd.read_csv(f"{path_curr}/GSEA(ebayes)_group_orgn_limma.csv", index_col=0)
df_dmps["CpG"] = df_dmps.index.values
df_dmps.sort_values(["adj.P.Val"], ascending=[True], inplace=True)
df_dmps['print'] = df_dmps.apply(lambda row: f"{row['CpG'].split('_')[0]}", axis=1)
df_dmps['log_pval'] = -np.log10(df_dmps["adj.P.Val"])

sns.set_theme(style='whitegrid')
volcano(
    df=df_dmps,
    lfc='logFC',
    pv='adj.P.Val',
    pv_thr=(0.05, 0.05),
    lfc_thr=(0.0, 0.0),
    path=f"{path_curr}",
    geneid='print',
    axtickfontsize=12,
    axlabelfontsize=12,
    gfont=18,
    gstyle=2,
    sign_line_h=True,
    sign_line_v=False,
    ar=0,
    color=('green', 'gray', 'red'),
    dim=(4, 4)
)

In [ ]:
path = f"D:/Yandex.Disk/pydnameth/draft/13_fmba_cvd_dnam/data/56_vs_56"
pheno = pd.read_excel(f"{path}/pheno_funnorm.xlsx", index_col=0)
pheno.index = pheno.index.astype(str)
betas = pd.read_pickle(f"{path}/betas_funnorm.pkl")
feats_beta = ['cg26769493_BC21', 'cg02601188_TC21', 'cg17296940_TC21', 'cg10060521_TC21', 'cg10864794_TC21']
betas = betas[feats_beta]

feats_pheno = ['Age', 'Status']
pheno = pheno[feats_pheno]

df_for_plot = pd.merge(pheno, betas, left_index=True, right_index=True)

pheno_associations = {
    'Status': {
        'groups': ['Control', 'Case'],
        'base': 'Control',
        'colors': {'Control': 'chartreuse', 'Case': 'red'}
    }
}

xy_min = df_for_plot['Age'].min().min()
xy_max = df_for_plot['Age'].max().max()
xy_ptp = xy_max - xy_min

for cpg in feats_beta:
    scatter = sns.scatterplot(
        data=df_for_plot,
        x='Age',
        y=cpg,
        hue='Status',
        palette=pheno_associations['Status']['colors'],
        linewidth=0.5,
        alpha=0.75,
        edgecolor="k",
        s=20,
        hue_order=list(pheno_associations['Status']['colors'].keys()),
        legend=True,
    )
    plt.savefig(f"{path}/scatter_{cpg}.png", bbox_inches='tight', dpi=400)
    plt.savefig(f"{path}/scatter_{cpg}.pdf", bbox_inches='tight', dpi=400)
    plt.clf()
    plt.close()

    violin = sns.violinplot(
        data=df_for_plot,
        x='Status',
        y=cpg,
        hue='Status',
        palette=pheno_associations['Status']['colors'],
        density_norm='width',
        order=pheno_associations['Status']['groups'],
        saturation=0.75,
        linewidth=1.0,
        legend=False,
        cut=0,
    )
    plt.savefig(f"{path}/violin_{cpg}.png", bbox_inches='tight', dpi=400)
    plt.savefig(f"{path}/violin_{cpg}.pdf", bbox_inches='tight', dpi=400)
    plt.clf()
    plt.close()

In [ ]:
fig = plt.figure(
        figsize=(20, 36),
        layout="constrained"
    )

axs = fig.subplot_mosaic(
    [
        ['21', '31'],
        ['22', '32'],
        ['23', '33'],
        ['24', '34'],
        ['25', '35'],
    ],
    height_ratios=[1, 1, 1, 1, 1],
    width_ratios=[1, 1],
)

sns.set_theme(style='whitegrid')

for id, cpg in enumerate(feats_beta):
    row_id = id % 5 + 1
    column_id = id % 2 + 2
    violin = sns.violinplot(
        data=df_for_plot,
        x='Status',
        y=cpg,
        hue='Status',
        palette=pheno_associations['Status']['colors'],
        density_norm='width',
        order=pheno_associations['Status']['groups'],
        saturation=0.75,
        linewidth=1.0,
        legend=False,
        ax=axs[f'{column_id}{row_id}'],
        cut=0,
    )
plt.savefig(f"{path}/DMPs_cpg.png", bbox_inches='tight', dpi=400)
plt.savefig(f"{path}/DMPs_cpg.pdf", bbox_inches='tight', dpi=400)

In [ ]:
def process_str_elem(x, delimiter: str = ';', missed = ''):
    if isinstance(x, str):
        elems = x.split(';')
        elems = list(set(elems))
        elems = delimiter.join(elems)
    else:
        elems = missed
    return elems

df_mnfst = pd.read_pickle(f"D:/Yandex.Disk/pydnameth/datasets/GPL33022/manifest.pkl")
df_mnfst['UCSC_RefGene_Name'] = df_mnfst['UCSC_RefGene_Name'].apply(process_str_elem, missed='non-genic')

In [ ]:
fig = plt.figure(
        figsize=(15, 10),
        layout="constrained"
    )
"""
axs = fig.subplot_mosaic(
    [
        ['1', '21', '31'],
        ['1', '22', '32'],
        ['1', '23', '33'],
        ['1', '24', '34'],
        ['1', '25', '35'],
        ['1', '26', '36'],
    ],
    height_ratios=[1, 1, 1, 1, 1, 1],
    width_ratios=[2, 1, 1],
)
"""
sns.set_theme(style='ticks')

axs = fig.subplot_mosaic(
    [
        ['1', '.', '21', '31'],
        ['1', '.', '22', '32'],
        ['1', '.', '23', '33'],
        ['1', '.', '24', '34'],
        ['1', '.', '25', '35'],
    ],
    height_ratios=[1, 1, 1, 1, 1],
    width_ratios=[4, 0.05, 1, 1],
    gridspec_kw={
                "bottom": 0.14,
                "top": 0.95,
                # "left": 0.1,
                # "right": 0.5,
                "wspace": 0.1,
                "hspace": 0.07,
            },
)
volc = volcano(
    df=df_dmps,
    lfc='logFC',
    pv='adj.P.Val',
    pv_thr=(0.05, 0.05),
    lfc_thr=(0.0, 0.0),
    path=f"{path_curr}",
    geneid='print',
    axtickfontsize=18,
    axlabelfontsize=18,
    gfont=18,
    gstyle=2,
    sign_line_h=True,
    sign_line_v=False,
    ar=0,
    color=('green', 'gray', 'red'),
    dim=(4, 4), 
    ret=True,
    ax=axs['1'],
    dotsize=5,
)
axs['1'].grid(True)
#axs['1'].text(-0.33, 1.5, 'A', fontsize=30, fontfamily='arial')
rect = Rectangle((0.01, 1.28), 0.09, 0.33, linewidth=2, edgecolor='r', facecolor='none')
axs['1'].add_patch(rect)

for id, cpg in enumerate(feats_beta):
    row_id = id % 5 + 1
    column_id = id % 2 + 2
    violin = sns.violinplot(
        data=df_for_plot,
        x='Status',
        y=cpg,
        hue='Status',
        palette=pheno_associations['Status']['colors'],
        density_norm='width',
        order=pheno_associations['Status']['groups'],
        saturation=0.75,
        linewidth=1.0,
        legend=False,
        ax=axs[f'{column_id}{row_id}'],
        cut=0,
    )
    axs[f'{column_id}{row_id}'].set(xlabel=None)
    axs[f'{column_id}{row_id}'].set(ylabel=f"{cpg}\n({df_mnfst.at[cpg, 'UCSC_RefGene_Name']})")
#axs['21'].text(-1.8, 0.79, 'B', fontsize=30, fontfamily='arial')

plt.savefig(f"{path}/volcano_with_hypo.png", bbox_inches='tight', dpi=400)
plt.savefig(f"{path}/volcano_with_hypo.pdf", bbox_inches='tight', dpi=400)